# 08_00 Advanced: fine-tuning a pretrained model (overview)

*📖 Book: section 5.5, chapters 6 and 7, and appendix E*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: a borrowed model, and a new job for it
>
> Project files this connects to:
>
> - `src/homer_ai/gpt.py`
> - `src/homer_ai/pretrained.py`
> - `src/homer_ai/finetune.py`
> - `src/homer_ai/lora.py`
>
> **This notebook:** Building a model from scratch taught us how it works. In real work you mostly take a model somebody already trained and adapt it. What does that involve, and what can a laptop do?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- The four ways to get a model to do a job, from cheapest to dearest
- What fine-tuning is, and why we borrow GPT-2 for it instead of using our Homer model
- What freezing, a classification head, instruction fine-tuning and LoRA are, in one picture each
- What this chapter downloads, and what results to expect on a laptop

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **pretrained model** | a model somebody already trained, shared as a file of weights | GPT-2 small: one file of 548 MB |
| **GPT-2 small** | the smallest of the GPT-2 models OpenAI published in 2019: 12 blocks, 124 million weights, trained on English web pages | the model this chapter borrows |
| **RAG** | retrieval-augmented generation: a program first looks up text that fits the question and pastes it into the prompt, so the model answers from that text | a chatbot that searches your handbook before it answers |

Words you've met before: **fine-tuning** ([07_00](../07_advanced_iliad_and_combining/07_00_chapter_overview.ipynb)).

## 3. The idea

### ❓ I want a model to do one job for me. What are my options?

There are four, and it helps to see them as a ladder. Each step up costs more, so you climb only as far as you need.

```
4. train from scratch   your own model, your own data          weeks, a lot of data
3. fine-tune            a pretrained model + your examples      minutes to hours
2. RAG                  look up the right text, paste it in     no training
1. prompt               just ask a big model                    no training
```

**Prompting** is asking well. **RAG** (retrieval-augmented generation) means a program looks up the text that fits the question and puts it in the prompt, so the model answers from it. **Fine-tuning** changes the weights of a model somebody already trained. **From scratch** is what chapters 1 to 7 did.

We started at the top of the ladder on purpose, because that's how you learn what's inside. In real work you start at the bottom. This chapter is step 3.

### ❓ So what is fine-tuning, really?

Here's the picture I keep in my head.

Our Homer model is like a child we taught to babble Greek from scratch. GPT-2 is an adult who already speaks English. **Fine-tuning** is giving that adult a short course for one new job, say "spot spam messages". That's far cheaper than raising a new child for it.

The rest of the chapter fits the same picture:

| In the picture | In the model | Notebook |
|---|---|---|
| the adult who already speaks English | GPT-2's weights, loaded into our own `GPTModel` | [08_01](08_01_loading_gpt2.ipynb) |
| they keep everything they know and learn only the new bit | **freezing**: most weights are locked, a few are trained | [08_02](08_02_spam_classifier.ipynb) |
| we teach them to take requests politely | **instruction fine-tuning**: examples of a request and its answer | [08_03](08_03_instructions.ipynb) |
| they take notes on a small notepad instead of rewiring their brain | **LoRA**: the weights stay locked, two thin tables beside them are trained | [08_04](08_04_lora.ipynb) |

The picture has a limit, and I'll say it once. An adult understands the job. GPT-2 doesn't. It's still only guessing the next token, and fine-tuning just shifts those guesses. So when it fails, it doesn't fail like a person would. You'll see that in 08_03.

### ❓ Why not fine-tune our own Homer model?

Because there's almost nothing in it to build on. Fine-tuning works when the model already knows a lot, and the short course only has to point that knowledge at a job.

Our GPT read 17,730 characters of one book of Greek. GPT-2 read about 40 GB of English web pages. Let's put the two side by side. The next cell builds both models empty (no download yet), just to count.

👉 **Predict first:** our GPT has 835,072 weights. How many times bigger is GPT-2 small: 10 times, 100 times, more?

In [ ]:
from homer_ai.gpt import GPTConfig, GPTModel
from homer_ai.pretrained import GPT2_CONFIG  # GPT-2 small, written as our own GPTConfig

ours = GPTConfig(vocab_size=137)  # the GPT of chapter 4


def count(config):
    return sum(p.numel() for p in GPTModel(config).parameters())


print(f"{'':16} {'our GPT':>12} {'GPT-2 small':>12}")
for name in ("vocab_size", "context_length", "emb_dim", "n_heads", "n_layers"):
    print(f"{name:16} {getattr(ours, name):>12,} {getattr(GPT2_CONFIG, name):>12,}")
n_ours, n_gpt2 = count(ours), count(GPT2_CONFIG)
print(f"{'weights':16} {n_ours:>12,} {n_gpt2:>12,}")
print(f"GPT-2 small is {n_gpt2 / n_ours:.0f} times bigger")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About 195 times bigger: 163,037,184 weights against 835,072. Every row is the same *kind* of setting, though. GPT-2 small is our `GPTModel` with bigger numbers: 12 blocks where we have 4, 768 numbers per token where we have 128. That's why its weights fit into our class, which is what [08_01](08_01_loading_gpt2.ipynb) does.

(You may know GPT-2 small as "124 million". Built our way it has 163 million, because GPT-2 uses one table twice and we keep two copies. 08_01 counts it out.)

</details>

### ❓ Why GPT-2, of all models?

Three reasons. It's small enough for a laptop: one file of 548 MB, and it runs on a CPU. OpenAI published its weights under a modified MIT licence, so anyone may use them. And it's the model our chapter 4 copies, block for block, so there's nothing new to learn before we can load it.

It's also old (2019) and weak next to today's models. That's fine for us. The steps are the same for a big model, they just need a bigger computer.

### ❓ What do I need on my computer?

- **One download of 548 MB**, GPT-2's weights. It goes to a cache folder outside the project, `~/.cache/homer-ai/gpt2/`, and is checked after downloading. 08_01 does it; the later notebooks reuse the file.
- **One download of 0.2 MB**, the text messages for 08_02 and 08_04, to `~/.cache/homer-ai/sms_spam/`.
- About **2 GB of free memory** while a notebook runs.
- A few minutes per notebook. On my MacBook (M1 Pro, CPU only) the slowest one takes about two and a half minutes.

Nothing is written into the project folder, and your Homer models aren't touched. The helper code lives in `src/homer_ai/`: `pretrained.py`, `finetune.py` and `lora.py`, imported like the rest of the project.

### ❓ How good will the results be?

Mixed, and I'd rather tell you now. These are the numbers from my laptop:

| Notebook | What we measure | Result |
|---|---|---|
| 08_01 | does GPT-2 write English from inside our own class? | yes, fluent English |
| 08_02 | spam or not, on 300 messages the model never saw | 93.7% right, after about a minute of training |
| 08_03 | three small text jobs given as instructions | the easy one 28 of 30, the two harder ones 15 of 30 each |
| 08_04 | spam again, with LoRA | 94.3% right, with 5 times fewer trained weights, but each step is slower |

So sorting text works well, even on a CPU. Following instructions works only for the easiest job. GPT-2 small is too small, and a minute of training is too short, for more than that.

### ❓ Which notebook do I open for each idea?

| Notebook | Question it answers |
|---|---|
| [`08_01_loading_gpt2.ipynb`](08_01_loading_gpt2.ipynb) | What's inside a weights file, and how do GPT-2's numbers get into our model? |
| [`08_02_spam_classifier.ipynb`](08_02_spam_classifier.ipynb) | How do I turn a model that writes into one that sorts messages? |
| [`08_03_instructions.ipynb`](08_03_instructions.ipynb) | How do I teach a model to answer a request, and how far does a small one get? |
| [`08_04_lora.ipynb`](08_04_lora.ipynb) | How do I fine-tune by training far fewer numbers? |

## 4. Recap

- Prompt, RAG, fine-tune, train from scratch: four ways to get a job done, each dearer than the one before.
- Fine-tuning is a short course for a model that already knows a lot. Our Homer model knows too little, so we borrow GPT-2 small.
- GPT-2 small is our `GPTModel` with bigger settings, so its weights fit into our own class.
- Freezing, instruction fine-tuning and LoRA are three ways to run that short course.

**Next:** [08_01: Loading GPT-2's weights by hand](08_01_loading_gpt2.ipynb).